[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch3/lesson2.ipynb)

# Observar más de cerca mediante satélites

Esta lección muestra cómo utilizar Earth Engine y geemap para encontrar y visualizar la imagen satelital disponible con menor cobertura de nubes sobre un lugar y cerca de una fecha determinada. Al ajustar los filtros y ordenar los resultados, puedes mostrar la imagen más útil, con la menor cantidad posible de nubes que oculten la superficie.

In [ ]:
import folium
import ee
import geemap
from datetime import datetime, timedelta

# Autenticar tu cuenta de Google con Earth Engine
ee.Authenticate()

# Escribe aquí el ID de tu proyecto, entre comillas
ee.Initialize(project="emerge-lessons")

In [ ]:
# Definir el punto de interés
latitude = 27.536873
longitude = -81.469549

map = folium.Map(
    location=[latitude, longitude],
    tiles="Cartodb dark_matter",
    zoom_start=9
)

Nota: En la celda anterior utilizamos la biblioteca de Python `folium` para crear el mapa. Otra opción sería utilizar lo siguiente:

```python
Map = geemap.Map(center=(lat, lon), zoom=10)
```

El resultado sería similar. Utilizamos `folium` en lugar de `geemap` porque permite que el mapa continúe visible después de publicar el cuaderno en línea, mientras que el método `geemap.Map()` puede ser más difícil de adaptar al formato del sitio.

Al igual que en el capítulo 1, debemos definir una función, adaptada de [este tutorial](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api), para agregar datos de Google Earth Engine a un mapa y mostrarlos de manera interactiva.

In [ ]:
def add_ee_layer(self, ee_image_object, vis_params, name):
    """
    Agrega a un mapa de Folium una capa de teselas
    procedente de una imagen de Earth Engine.
    """
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)

    folium.raster_layers.TileLayer(
        tiles=map_id_dict["tile_fetcher"].url_format,
        attr=(
            'Map Data &copy; '
            '<a href="https://earthengine.google.com/">'
            "Google Earth Engine</a>"
        ),
        name=name,
        overlay=True,
        control=True
    ).add_to(self)


folium.Map.add_ee_layer = add_ee_layer

In [ ]:
# Crear un punto a partir de la longitud y la latitud
point = ee.Geometry.Point([longitude, latitude])

# Elegir la fecha objetivo a la que queremos acercarnos
target_date_str = "2024-01-01"
target_date = datetime.strptime(target_date_str, "%Y-%m-%d")

# Definir una ventana temporal, por ejemplo, 60 días antes y después
# Puedes ampliarla o reducirla según la cantidad de imágenes
# que quieras considerar
window_days = 60

start_date = (
    target_date - timedelta(days=window_days)
).strftime("%Y-%m-%d")

end_date = (
    target_date + timedelta(days=window_days)
).strftime("%Y-%m-%d")

In [ ]:
# Crear una colección de imágenes Sentinel-2
collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(point)  # Incluir solo imágenes sobre el punto
    .filterDate(start_date, end_date)  # Aplicar la ventana de fechas
    .sort(
        "CLOUDY_PIXEL_PERCENTAGE"
    )  # Colocar primero la imagen con menos nubes
)

# Obtener la imagen con menos nubes dentro de la ventana
image = collection.first()

In [ ]:
# Agregar la imagen al mapa solamente si se encontró una
if image.getInfo():
    # Obtener la fecha de captura de la imagen
    image_date = (
        ee.Date(image.get("system:time_start"))
        .format("YYYY-MM-dd")
        .getInfo()
    )

    # Obtener el porcentaje de nubosidad
    cloud_pct = image.get(
        "CLOUDY_PIXEL_PERCENTAGE"
    ).getInfo()

    # Parámetros de visualización en color verdadero
    vis_params = {
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 3000
    }

    # Agregar la imagen con menos nubes
    # El nombre de la capa muestra la fecha y el porcentaje de nubosidad
    map.add_ee_layer(
        image,
        vis_params,
        (
            f"Menor nubosidad: {image_date} "
            f"({cloud_pct:.2f} % de nubes)"
        )
    )

# Mostrar el mapa
display(map)